# Setup


In [ ]:

%pip install -q -U "crewai[google-genai]"


# Auth & Flow


In [ ]:

import os
from google.colab import userdata

from crewai import LLM
from crewai.flow.flow import Flow, start, listen, router
from pydantic import BaseModel, Field
from crewai.flow.flow import or_


GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")


# Ticket state


In [ ]:

class TicketState(BaseModel):
    ticket_text: str = ""
    category: str = ""
    priority: str = ""
    resolution: str = ""


# Support flow


In [ ]:

class CustomerSupportFlow(Flow[TicketState]):

    @start()
    def receive_ticket(self):
        # Enter customer issue (e.g. billing charge question, login MFA reset, feature request)
        ticket = input("Enter customer support issue: ").strip()

        if not ticket:
            raise ValueError("Support ticket cannot be empty.")

        self.state.ticket_text = ticket
        print("\nTicket received successfully!")
        print("Ticket:", self.state.ticket_text)

    @listen(receive_ticket)
    def categorize_ticket(self):
        prompt = f"""
        You are a customer support ticket classifier.

        Classify the following customer ticket into exactly ONE category:
        - billing: payments, refunds, invoices, charges
        - account_access: login problems, passwords, account lockouts
        - other: technical issues, feature requests, all other problems

        Return ONLY the category name.
        Do not explain your answer.

        Customer ticket:
        {self.state.ticket_text}
        """

        response = llm.call(prompt)
        category = str(response).strip().lower()

        if "billing" in category:
            category = "billing"
        elif "account_access" in category or "account access" in category:
            category = "account_access"
        else:
            category = "other"

        self.state.category = category

        print("\nTicket categorized!")
        print("Category:", self.state.category)

    @router(categorize_ticket)
    def route_by_category(self):
        category = self.state.category

        print(f"\nRouting ticket to: {category}")

        if category == "billing":
            return "billing"
        elif category == "account_access":
            return "account_access"
        else:
            return "other"

    @listen("billing")
    def handle_billing(self):
        self.state.priority = "High"
        self.state.resolution = (
            "Billing Support: Verify the transaction, review the invoice, "
            "and investigate any refund or incorrect charge."
        )
        print("Assigned to Billing Support.")

    @listen("account_access")
    def handle_access(self):
        self.state.priority = "High"
        self.state.resolution = (
            "Account Access Support: Help the customer reset their "
            "password, resolve login problems, or unlock their account."
        )
        print("Assigned to Account Access Support.")

    @listen("other")
    def handle_other(self):
        self.state.priority = "Medium"
        self.state.resolution = (
            "General Support: Investigate the technical issue or feature "
            "request and provide troubleshooting or escalation steps."
        )
        print("Assigned to General Support.")

    @listen(or_(handle_billing, handle_access, handle_other))
    def resolve_ticket(self):
        print("\n========== TICKET RESOLUTION ==========")
        print("Ticket:", self.state.ticket_text)
        print("Category:", self.state.category)
        print("Priority:", self.state.priority)
        print("Resolution:", self.state.resolution)
        print("=======================================")

        return {
            "ticket": self.state.ticket_text,
            "category": self.state.category,
            "priority": self.state.priority,
            "resolution": self.state.resolution
        }

support_flow = CustomerSupportFlow()
result = support_flow.kickoff()
print(result)




